# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhaledObeid/KhaledObeid-flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: declining-content triage.** FlyRank's product already flags declining pages with hand-written rules (health score, needs-attention tags) — the rules work, but they don't say which flagged page an editor should fix *first* when there are hundreds waiting and only a few hours in the week. That prioritization is exactly where messy, tangled signals (traffic volume, position, keyword competition, freshness, content type) make a plain if-statement fall short, and where a learned ranking can beat the current rule. I'm picking this lane over freestyle because it's the one this internship's whole skill chain (baseline → model → validation → playbook) is built to answer, and the data in hand is shaped for it.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

per_client = df.groupby("client_id").agg(
    pages=("content_id", "count"),
    declining_pages=("trend_direction", lambda s: (s == "down").sum()),
)

print(f"{len(df):,} pages across {df['client_id'].nunique()} clients")
print(
    f"clients with 100+ pages already flagged as declining: "
    f"{(per_client['declining_pages'] >= 100).sum()} of {len(per_client)}"
)
per_client["declining_pages"].describe()[["min", "25%", "50%", "75%", "max"]]

30,000 pages across 32 clients
clients with 100+ pages already flagged as declining: 19 of 32


min       0.00
25%      42.00
50%     215.50
75%     534.25
max    3435.00
Name: declining_pages, dtype: float64

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** out of all the pages a client already has flagged as declining, which one should a FlyRank content editor pick up first this week.

**Actor:** the content editor / account team — they'd use a ranked queue as a work list, not as something that changes a page on its own.

**Cost of a wrong call is asymmetric.** Under-ranking a high-traffic page that's declining means real clicks and impressions keep bleeding away for another week before anyone touches it — that's lost visibility a client notices. Over-ranking a low-traffic or noisy page just wastes an editor's limited hours on something that barely moves the client's numbers. Because editor time is the scarce resource, the expensive mistake is missing the big decliners, not double-checking a small one.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

worst_client = per_client["declining_pages"].idxmax()
worst_backlog = int(per_client.loc[worst_client, "declining_pages"]) # type: ignore[arg-type]

worst_impressions = df.loc[
    (df["client_id"] == worst_client) & (df["trend_direction"] == "down"), "impressions_90d"
].sum()

print(f"median declining backlog per client: {int(per_client['declining_pages'].median())} pages")
print(
    f"largest single-client backlog: {worst_backlog} declining pages, "
    f"{worst_impressions:,.0f} impressions/90d at stake"
)
print("that's the queue one editor faces on one client in one week — 'which one first' is a real weekly decision.")

median declining backlog per client: 215 pages
largest single-client backlog: 3435 declining pages, 24,728,273 impressions/90d at stake
that's the queue one editor faces on one client in one week — 'which one first' is a real weekly decision.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**Three numbers that justify the lane:** 54.2% of the 30,000 pages in this slice are already trending down; of the 73.4% of pages with enough traffic to matter (≥100 impressions/90d and at least one session), 59.8% are declining; and those measurable, declining pages alone hold 51.2% of the slice's total 90-day impressions. Roughly half of all the search visibility in this dataset sits on pages that are currently losing ground — that's not a rare edge case to patch, it's the scale where "which one first" becomes a real, costly decision rather than a rounding error.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

declining = df["trend_direction"] == "down"
measurable = (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)

pct_declining = declining.mean() * 100
pct_measurable = measurable.mean() * 100
pct_declining_in_measurable = declining[measurable].mean() * 100

impressions_declining_measurable = df.loc[declining & measurable, "impressions_90d"].sum()
impressions_total = df["impressions_90d"].sum()
share_of_visibility = impressions_declining_measurable / impressions_total * 100

print(f"declining (trend_direction == 'down'): {pct_declining:.1f}% of all pages")
print(f"measurable traffic (>=100 impressions/90d and >=1 session): {pct_measurable:.1f}% of all pages")
print(f"declining, among measurable-traffic pages: {pct_declining_in_measurable:.1f}%")
print(f"share of ALL 90-day impressions sitting on declining+measurable pages: {share_of_visibility:.1f}%")

declining (trend_direction == 'down'): 54.2% of all pages
measurable traffic (>=100 impressions/90d and >=1 session): 73.4% of all pages
declining, among measurable-traffic pages: 59.8%
share of ALL 90-day impressions sitting on declining+measurable pages: 51.2%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can claim:** an *observed* trend direction over each page's trailing 90-day window, computed directly from GSC impressions, not guessed; a *directional* signal showing which flagged pages share traits with past decliners; a *decision-support* priority queue that helps an editor spend limited hours on the pages losing the most real traffic.

**Can't claim:** that refreshing a page *causes* it to recover (that needs a controlled before/after comparison, not a snapshot); that the model predicts Google's ranking algorithm; or that a score here is a verdict rather than a suggestion an editor can override. `trend_direction` and `trend_pct` are the source of the label itself, so they will never be model inputs later — using them as features would just teach the model to repeat the rule that already labeled the data, not to find something new.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

label_source_check = pd.crosstab(df["trend_direction"], df["trend_direction"] == "down")
print(label_source_check)
print()
print("is_declining_label is exactly (trend_direction == 'down') relabeled —")
print("so trend_direction and trend_pct can inform this RESEARCH QUESTION,")
print("but must never be model features later (that would be learning the rule, not the world).")

trend_direction  False  True 
trend_direction              
down                 0  16262
flat              1152      0
new               2236      0
stable            5962      0
up                4388      0

is_declining_label is exactly (trend_direction == 'down') relabeled —
so trend_direction and trend_pct can inform this RESEARCH QUESTION,
but must never be model features later (that would be learning the rule, not the world).


: 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.